# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)



## 1 Two Paper Findings + My Methodology Questions
Finding 1

The FlyRank research paper reports that the machine learning analysis used Random Forest, Logistic Regression, Decision Tree, K-Means, and PCA on a feature-vector dataset. The paper also states that these machine learning results are exploratory and secondary to the direct portfolio analysis.

My methodology question

How were the training and testing datasets separated? A grouped or time-aware validation strategy would reduce information leakage and provide a more reliable estimate of model performance.

Finding 2

The paper states that the study is observational and that correlations do not prove causation. It also explains that the Health Score is a FlyRank composite metric rather than a Google ranking metric.

My methodology question

How sensitive are the reported findings to different feature definitions and validation strategies? Testing multiple validation approaches could help determine whether the observed relationships remain consistent.

In [11]:
# =====================================================
# ML-09 Setup
# Load Data and Prepare Features
# =====================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GroupShuffleSplit
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42

# -----------------------------
# Load dataset
# -----------------------------

DATA_PATH = "/content/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

# -----------------------------
# Create target
# -----------------------------

df["target"] = (
    df["trend_direction"] == "down"
).astype(int)

# -----------------------------
# Remove leakage columns
# -----------------------------

drop_cols = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct",
    "target",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

X = df.drop(columns=drop_cols)

X = pd.get_dummies(X, dummy_na=True)
X = X.fillna(0)

y = df["target"]

groups = df["client_id"]

print("Dataset Ready")
print(X.shape)

Dataset Ready
(30000, 76)


In [12]:
# ==========================================
# Section 1 Code
# Research Paper Audit
# ==========================================

paper_findings = {
    "Finding 1": "ML results are exploratory and use Random Forest, Logistic Regression, Decision Tree, PCA and K-Means.",
    "Question 1": "Would grouped or time-aware validation produce similar results?",

    "Finding 2": "The study is observational and correlations do not imply causation.",
    "Question 2": "Would the conclusions remain consistent under different validation designs?"
}

for key, value in paper_findings.items():
    print(f"{key}:")
    print(value)
    print()

Finding 1:
ML results are exploratory and use Random Forest, Logistic Regression, Decision Tree, PCA and K-Means.

Question 1:
Would grouped or time-aware validation produce similar results?

Finding 2:
The study is observational and correlations do not imply causation.

Question 2:
Would the conclusions remain consistent under different validation designs?



## 2. My model under an honest split (before/after)



To evaluate whether the validation strategy affects model performance, I compared two train/test splitting methods.

The first experiment used a standard random train/test split. This approach is simple but may allow information from the same client to appear in both the training and testing data.

The second experiment used the grouped split from ML-08, where all pages from the same client remain in either the training or testing set.

The grouped split provides a more realistic estimate of model performance because it reduces information leakage between clients. The comparison below shows the model performance under both validation strategies.

In [13]:
# =====================================================
# Section 2
# My Model Under an Honest Split (Before vs After)
# =====================================================

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# =====================================================
# BEFORE
# Standard Random Split
# =====================================================

X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

rf_random = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_random.fit(X_train_random, y_train_random)

pred_random = rf_random.predict(X_test_random)

random_accuracy = accuracy_score(y_test_random, pred_random)
random_f1 = f1_score(y_test_random, pred_random)

# =====================================================
# AFTER
# Honest Grouped Split
# =====================================================

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

rf_group = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_group.fit(X_train, y_train)

pred_group = rf_group.predict(X_test)

group_accuracy = accuracy_score(y_test, pred_group)
group_f1 = f1_score(y_test, pred_group)

# =====================================================
# Comparison Table
# =====================================================

comparison = pd.DataFrame({
    "Validation": ["Random Split", "Grouped Split"],
    "Accuracy": [random_accuracy, group_accuracy],
    "F1 Score": [random_f1, group_f1]
})

print("=" * 50)
print("Validation Comparison")
print("=" * 50)

display(comparison.round(3))

Validation Comparison


,Validation,Accuracy,F1 Score
0,Random Split,0.700,0.734
1,Grouped Split,0.572,0.594


## 3. Leakage audit


The final feature set was reviewed to identify possible sources of target leakage.

Columns directly used to create the target variable (trend_direction and trend_pct) were removed before model training. In addition, recent performance columns used to derive the trend label (impressions_last_30d, clicks_last_30d, sessions_last_30d, impressions_prev_30d, clicks_prev_30d, and sessions_prev_30d) were excluded from the feature set.

The remaining features describe page characteristics, engagement metrics, and historical performance that would reasonably be available before making a refresh decision.

The audit found no evidence that the model was trained using the target label or variables directly derived from it. Therefore, the final feature set represents an honest modeling approach suitable for decision-support.

In [14]:
# =====================================================
# Section 3
# Leakage Audit
# =====================================================

print("=" * 60)
print("Leakage Audit")
print("=" * 60)

removed_columns = [
    "trend_direction",
    "trend_pct",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d"
]

print("\nColumns intentionally removed to prevent leakage:")

for col in removed_columns:
    print("-", col)

print("\nChecking for leakage columns remaining in feature matrix...")

leakage_found = False

for col in X.columns:
    if "trend" in col.lower():
        leakage_found = True
        print("Potential leakage:", col)

if not leakage_found:
    print("\nPASS: No trend-related columns found.")

print("\nFeature Matrix Shape:", X.shape)
print("Target Distribution:")
print(y.value_counts())

print("\nLeakage audit completed successfully.")

Leakage Audit

Columns intentionally removed to prevent leakage:
- trend_direction
- trend_pct
- impressions_last_30d
- clicks_last_30d
- sessions_last_30d
- impressions_prev_30d
- clicks_prev_30d
- sessions_prev_30d

Checking for leakage columns remaining in feature matrix...

PASS: No trend-related columns found.

Feature Matrix Shape: (30000, 76)
Target Distribution:
target
1    16262
0    13738
Name: count, dtype: int64

Leakage audit completed successfully.


## 4. Claim rewrite

In ML-08, a strong claim could be:

> "The Random Forest model identifies pages that should be refreshed."

This statement goes beyond what the analysis can support because the model was trained on historical observational data rather than a controlled experiment.

A safer and more appropriate research claim is:

> "The Random Forest model showed higher predictive performance than the Week-5 rule-based baseline on the grouped validation split. The observed relationships are directional and should be used as decision-support rather than evidence of causal ranking improvements."

This revised claim reflects the limitations of the data and follows the recommended public-safe research language.

In [15]:
# =====================================================
# Section 4 Code
# Claim Rewrite
# =====================================================

original_claim = "The Random Forest model identifies pages that should be refreshed."

revised_claim = (
    "The Random Forest model showed higher predictive performance "
    "than the Week-5 rule-based baseline on the grouped validation split. "
    "The findings are observational, directional, and intended for "
    "decision-support rather than proving causal ranking improvements."
)

print("Original Claim:\n")
print(original_claim)

print("\n" + "="*70 + "\n")

print("Revised Research Claim:\n")
print(revised_claim)

Original Claim:

The Random Forest model identifies pages that should be refreshed.


Revised Research Claim:

The Random Forest model showed higher predictive performance than the Week-5 rule-based baseline on the grouped validation split. The findings are observational, directional, and intended for decision-support rather than proving causal ranking improvements.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.